# 30 · Two-by-two table, checked against statsmodels

## What this checks

**Two-by-two table** compares the risk of an outcome in two groups. This check covers release 0.8.3. It reports:

- **the risk in each group**, the share of people who had the outcome;
- **the risk ratio**, how many times higher one group's risk is;
- **the odds ratio**, the same comparison made with odds instead of risks;
- **the risk difference**, one risk minus the other.

Each comes with a 95% confidence interval: the range of values the data are consistent with.

For the people of sub-districts supplied by the Southwark and Vauxhall Company only, Fieldwork reports a risk ratio of
cholera death of **5.35** against Lambeth only. The interval is **3.36 to 8.53**. Does an independent implementation
get the same numbers?

**Nobody types the table in.** Fieldwork makes it, in the South London example's people workflow, from Snow's
Tables VIII and XII. This check reads the table that workflow hands to the Two-by-two table, so a change to the
workflow changes what is checked.

## How it is checked

**The four cells are built here.** We add up the rows of that table ourselves: deaths and survivors for each of the
two water supplies compared. We do not take the cells from Fieldwork.

**The estimates come from statsmodels**, a Python statistics library by other authors. It has the same methods,
written independently:

- **The risk ratio** uses the Katz log interval.
- **The odds ratio** uses the Wald log interval.
- **The risk difference** is missing from statsmodels' two-by-two tool. So it is compared against statsmodels'
  interval for a difference of two proportions, the unpooled Wald method.

A second method, Agresti-Caffo, is shown beside the risk difference. It gives a different interval for the same
quantity. The gap shows what Fieldwork's choice of method costs. We do not require it to match.

statsmodels lays out its table with exposure down the rows and cases in the first column. Fieldwork's request names
the same layout. A table turned on its side would give a confident disagreement about nothing. So we check the
layout too.

The data are John Snow's 1855 figures on people: Table VIII, grouped by the water supply that Table XII declares.

## What should happen

1. **The cells.** The four cells built here from the table's rows equal Fieldwork's exactly. The same water supply is
   left out of the comparison on both sides.
2. **Every estimate** within 0.000000001 of statsmodels. Each is a ratio or a difference of two divisions, with
   nothing to approximate.
3. **Every interval bound** within 0.000000001, at the matching method. Check 27 once found the number 1.9600
   standing in for 1.959963984540054. A tolerance this tight catches that kind of defect, instead of letting it pass.
4. **The layout.** The risks recomputed from the cells equal the risks Fieldwork reports. A table on its side agrees
   on the odds ratio and disagrees on everything else, so this tells the two faults apart.
5. **Each label says where it was read.** Every estimate carries a label and its source. The source must be
   `results-panel`. A label that claims `original-program` fails, and so does a label with no source.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["statsmodels", "scipy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_30_two_by_two import run

report = run(Path.cwd() / "fixtures" / "two-by-two-snow-002.json")
print("agrees:", report["agrees"])
print(f"  orientation: {'OK' if report['orientation']['agrees'] else 'DIFF'}")
for e in report["estimates"]:
    f, s = e["fieldwork"], e["statsmodels"]
    print(f"  {'OK ' if e['agrees'] else 'DIFF'} {e['name']:16s} "
          f"{f['value']:.9f} [{f['lower']:.9f}, {f['upper']:.9f}]  "
          f"vs {s['value']:.9f} [{s['lower']:.9f}, {s['upper']:.9f}]")

## The result on record

First recorded on 27 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.8.3:
**Agrees**.

- **The cells:** 844 deaths and 166,810 survivors for Southwark and Vauxhall only; 18 deaths and 19,115 survivors for
  Lambeth only. Built here, they equal Fieldwork's. The both-companies group is left out on both sides.
- **The risk ratio:** 5.351051, from 3.355714 to 8.532834.
- **The odds ratio:** 5.373066, from 3.367904 to 8.572048.
- **The risk difference:** 0.004093, from 0.003543 to 0.004644.

Every estimate, every bound and every method name is exact. The layout was verified. The shared z value equals
scipy's within 0.000000000001. Every label is present, with `results-panel` as its source.

When first recorded, the check also covered a second table, on houses. The houses comparison is now made by the
Two-by-two table's rate layout, and check 27 covers it.

Running the cell above repeats the check on the same data.

## The four cells

The cell below sets the cells built here beside the ones Fieldwork counted, and names the water supply left out of
the comparison.

In [ ]:
# The four cells, built here from the table's rows, against the ones Fieldwork counted.
cells = report["cells"]
for name, value in cells["recomputed"].items():
    print(f"{name:18} {value:>8}   Fieldwork {cells['fieldwork'][name]:>8}")
print("left out of the comparison:", cells["leftOut"])
print("cells agree:", cells["agrees"])

## The label, and where it was read

Every estimate carries the words another program prints for the same quantity. That way a trained reader can find
the row they are looking for:

- **Risk ratio, Katz log**, is printed as `Risk Ratio (RR)`.
- **Odds ratio, Wald log**, is printed as `Odds Ratio (cross product)`.
- **Risk difference, unpooled Wald**, is printed as `Risk Difference (RD%)`.

These labels were **read from another program's published results panel**, not from the program itself. So the
labels are a deduction, not a verification. The source field on each estimate records which: `results-panel`.

**This check fails if a label ever claims `original-program` as its source**, because nobody has read the program
itself. It also fails on a bare label with no source. A field that can be filled without being verified is worse than
no field.

Note the scale. That panel prints the risk difference as a percentage. So Snow's `0.004093` reads there as
`0.4093%`. It is the same quantity on a different scale.

The cell below prints each label, its source, and the check's criterion for them.

In [ ]:
for e in report["estimates"]:
    print(f"{e['name']:16} {str(e['label']):28} {e['labelSource']}")

print("\nlabel problems:", report["labelProblems"])
print("criterion     :", report["criterion"]["labels"])

## What this does not show

- **That the labels are that program's own.** They were read from its published results panel. So they are a
  deduction about what it prints, not a reading of it.
- **That the comparison is fair.** The both-companies group is left out by name, not controlled for. That is Snow's
  own confounding. It is why he went down to the house level instead of settling it by sub-district.
- **That the normal approximations are good enough.** The Lambeth-only group has 18 deaths. An exact interval would be
  the better comparison there.
- **Which denominator is right.** Houses and people give 8.40 and 5.35 for the same outbreak. That choice is the
  practitioner's. Check 27 covers the houses comparison.